# 6.3 LSTM Text Classification

這份 Notebook 使用 LSTM 建立客服訊息三分類模型。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)


## 2. 建立客服意圖分類資料


In [ ]:
CLASS_NAMES = np.array(['billing', 'shipping', 'technical'])

def make_intent_data(repeats=90, seed=42):
    rng = np.random.default_rng(seed)
    templates = {
        'billing': [
            'my invoice has a wrong charge',
            'please refund the payment',
            'the billing amount is incorrect',
            'i need a receipt for my order',
            'my credit card was charged twice',
        ],
        'shipping': [
            'my package arrived late',
            'where is the delivery tracking number',
            'the shipment is delayed today',
            'please change my delivery address',
            'the courier lost my package',
        ],
        'technical': [
            'i cannot login to the app',
            'the website shows an error message',
            'the feature crashes after update',
            'my password reset link failed',
            'the dashboard does not load',
        ],
    }
    prefixes = ['hello', 'urgent', 'please help', 'support team', 'hi']
    texts, labels = [], []
    for label, class_name in enumerate(CLASS_NAMES):
        for _ in range(repeats):
            for sentence in templates[class_name]:
                texts.append(rng.choice(prefixes) + ' ' + sentence)
                labels.append(label)
    texts = np.array(texts, dtype=object)
    labels = np.array(labels, dtype='int64')
    idx = rng.permutation(len(labels))
    return texts[idx], labels[idx]

def split_texts(texts, labels):
    x_train, x_temp, y_train, y_temp = train_test_split(
        texts, labels, test_size=0.3, random_state=42, stratify=labels
    )
    x_valid, x_test, y_valid, y_test = train_test_split(
        x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )
    return x_train, x_valid, x_test, y_train, y_valid, y_test

texts, labels = make_intent_data(repeats=90, seed=3)
x_train, x_valid, x_test, y_train, y_valid, y_test = split_texts(texts, labels)
print(len(x_train), len(x_valid), len(x_test))
pd.DataFrame({'text': x_train[:6], 'label': CLASS_NAMES[y_train[:6]]})


## 3. 建立 TextVectorization


In [ ]:
MAX_TOKENS = 1500
SEQ_LEN = 14
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode='int',
    output_sequence_length=SEQ_LEN,
)
vectorizer.adapt(x_train)
print(vectorizer.get_vocabulary()[:25])


## 4. 建立 LSTM 文字分類模型


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Embedding(MAX_TOKENS, 32),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 5. 訓練與評估


In [ ]:
history = model.fit(x_train, y_train, validation_data=(x_valid, y_valid), epochs=14, batch_size=32, verbose=0)
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='LSTM text accuracy')
plt.ylim(0, 1.05)
plt.show()
y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
print(model.evaluate(x_test, y_test, verbose=0, return_dict=True))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


## 6. 預測新客服訊息


In [ ]:
new_texts = np.array(['please refund my payment', 'the delivery tracking is late', 'i cannot login after update'], dtype=object)
prob = model.predict(new_texts, verbose=0)
pd.DataFrame(prob, columns=CLASS_NAMES, index=new_texts)


## 7. 如何套用自己的資料

建立類別名稱與 label mapping，並將每筆文字對應到類別 index。若資料很少，LSTM 可能過擬合，可以先和 CNN 或 simple pooling baseline 比較。


## 8. 小結

LSTM 會依序讀取 token，適合需要文字順序資訊的分類任務。
